# Visualising and exploring the Hardcastle catalogue & dataset

This code shows some possibly helpful visualisations of the Hardcastle dataset of radio sources, which is used as the data source for this project before pre-processing. This dataset is formed by the code in `hardcastle_catalogue/image_downloading`, and is made by combining information from the Hardcastle et al. (2023) catalogue of radio-optical cross-matched sources, and the corresponding radio images from the LOFAR Two-metre Sky Survey (LoTSS) Data Release 2 (DR2).

Table of contents:
1. [The Hardcastle catalogue](#the-hardcastle-catalogue)
2. [The Hardcastle dataset](#the-hardcastle-dataset)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd
from tqdm import tqdm

import utils.paths as pths

<a id='HardcastleCatalogue'></a>
# The Hardcastle catalogue

The paper describing the catalogue is [Hardcastle et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023MNRAS.518..257H/abstract), and as a reference:

M. J. Hardcastle et al. The LOFAR Two-Metre Sky Survey: VI. Optical identifications for the second data release. Astronomy & Astrophysics, 678:A151, October 2023. ISSN 1432-0746. doi: 10.1051/0004-6361/202347333.

In [ ]:
# Loading the catalogue - we only care about the resolved sources in our processes
import hardcastle_catalogue.image_downloading.hardcastle_catalogue as hc
hdc = hc.HardcastleCatalogue(resolved_only=True)

You can also see how many resolved items exist, if you'd like (as you can see, only around 7.5% of the sources in the catalogue are resolved i.e., have an angular size larger than the beam width of LOFAR)

In [ ]:
_ = hc.HardcastleCatalogue(resolved_only=False)

## Header information

The catalogue contains many columns of information, but the most important for our purposes are the following:

| Column Name | Type    | Units    | Description                                                                  |
|-------------|---------|----------|------------------------------------------------------------------------------|
| RA          | float64 | deg      | Radio right ascension (mean position)                                        |
| DEC         | float64 | deg      | Radio declination (mean position)                                            |
| Total_flux  | float64 | mJy      | 144-MHz total flux density                                                   |
| Peak_flux   | float64 | mJy/beam | 144-MHz peak flux density                                                    |
| Isl_rms     | float64 | mJy/beam | rms noise in island                                                          |
| Resolved    | bool    |          | Boolean flag to indicate whether source is resolved                          |
| LAS         | float64 | arcsec   | Estimate of angular size, only valid for sources with Resolved == True       |
| z_best      | float64 | deg      | Spec-z if available and good, else photo-z if available and good, else blank |
| L_144       | float64 | deg      | Radio luminosity in W/Hz for alpha=0.7                                       |

You can see these specific options in the `Source` Enum in `hardcastle_catalogue/image_downloading/hardcastle_catalogue.py`, where I have designed it to convert more plaintext options (e.g., Luminosity) into the column headers so described.

In [ ]:
src = hc.Source
# print all the column names and their descriptions
for col in src:
    print(f"{col.name}: {col.value}")

We can use the `HardcastleCatalogue` class to easily access the data in the catalogue, and to perform some basic visualisations. For example, we can plot the distribution of redshifts for the resolved sources:

In [ ]:
# Plotting a histogram of the redshift distribution for the resolved sources in the Hardcastle catalogue
plt.hist(hdc.get_values(src.Redshift), bins=50, edgecolor='black')
plt.xlabel('Redshift (z)')
plt.ylabel('Number of sources')
plt.title('Redshift distribution of resolved sources in the Hardcastle catalogue')
plt.grid()
plt.show()

# The Hardcastle Dataset

We can also look at some images present in the full Hardcastle dataset, which has not yet been encapsulated in a class but anyway...

In [ ]:
# Load the dataset from the FITS file
print("Loading the Hardcastle dataset from FITS file...")
dataset_file_path = pths.DATASET_PARENT/'hardcastle_catalogue_with_images.fits'
catalogue_data = []
# Get the information from the Hardcastle catalogue
with fits.open(dataset_file_path) as hdul:
    # Remove the first two HDUs which are just Primary and the header table
    hdul = hdul[2:]

    # Extract the pixel values from each imageHDU
    for idx, hdu in enumerate(tqdm(hdul, desc="Extracting pixel values from Hardcastle dataset")):
        try:
            if isinstance(hdu.data, np.ndarray):
                catalogue_data.append({'index': idx, 'pixel_values': hdu.data})
            else:
                print(f"Unexpected data type for HDU {idx}: {type(hdu.data)}. Expected numpy array.")
                catalogue_data.append({'index': idx, 'pixel_values': np.nan})
        except Exception as e:
            print(f"Error loading Hardcastle dataset item {idx}: {e}")
            catalogue_data.append({'index': idx, 'pixel_values': np.nan})

# Initialise all other columns to default right now
catalogue_data = [{**item, 'broken': False, 'S/N_sigma': 0, 'edge_max': 0} for item in catalogue_data]

# Set up DataFrame columns
columns = ['index', 'pixel_values', 'broken', 'S/N_sigma', 'edge_max']  # Add more columns as needed for header information
dataset = pd.DataFrame(catalogue_data, columns=columns)

In [ ]:
# Plotting a random image from the dataset
random_index = np.random.choice(dataset.index)
random_image = dataset.loc[random_index, 'pixel_values']
plt.imshow(random_image, cmap='gray')
plt.colorbar()
plt.title(f"Random image from Hardcastle dataset (index: {random_index})")
plt.show()